#### Test d'extraction en Parquet

In [ ]:
import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import csv

# Paramètres
dataset_id = "economicref-france-sirene-v3"
url = f"https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/{dataset_id}/exports/csv"
params = {
    "where": (
        "etablissementsiege='oui' AND "
        "naturejuridiqueunitelegale IN ("
        "'Société à responsabilité limitée (sans autre indication)', "
        "'SAS, société par actions simplifiée', "
        "'Société à responsabilité limitée (SARL)'"
        ")"
    ),
    "delimiter": ";",
    "lang": "fr"
}
filename_parquet = "extraction_sas_sarl_2026.parquet"
chunk_size = 50000 

print(f"🚀 Extraction sécurisée via CSV Reader...")

try:
    with requests.get(url, params=params, stream=True) as r:
        r.raise_for_status()
        
        # On décode ligne par ligne pour ne pas saturer la RAM
        lines_iterator = r.iter_lines(decode_unicode=True)
        
        # 1. Lecture du Header
        header_line = next(lines_iterator)
        column_names = header_line.split(';')
        num_cols = len(column_names)

        # 2. Utilisation du CSV Reader (Gestion des guillemets pour éviter les décalages)
        reader = csv.reader(lines_iterator, delimiter=';', quoting=csv.QUOTE_MINIMAL)

        writer = None
        total_lignes = 0
        current_chunk = []

        for parts in reader:
            if not parts: continue
            
            # Correction de structure si nécessaire
            if len(parts) != num_cols:
                if len(parts) > num_cols:
                    parts = parts[:num_cols]
                else:
                    parts += [''] * (num_cols - len(parts))
            
            current_chunk.append(parts)
            
            if len(current_chunk) >= chunk_size:
                df_chunk = pd.DataFrame(current_chunk, columns=column_names)
                
                # Conversion en types appropriés pour le Parquet
                table = pa.Table.from_pandas(df_chunk)
                
                if writer is None:
                    writer = pq.ParquetWriter(filename_parquet, table.schema, compression='snappy')
                
                writer.write_table(table)
                total_lignes += len(current_chunk)
                print(f"📥 {total_lignes:,} lignes sécurisées...")
                current_chunk = []

        # Dernier bloc
        if current_chunk:
            df_chunk = pd.DataFrame(current_chunk, columns=column_names)
            table = pa.Table.from_pandas(df_chunk)
            if writer is None:
                writer = pq.ParquetWriter(filename_parquet, table.schema, compression='snappy')
            writer.write_table(table)
            total_lignes += len(current_chunk)

        if writer:
            writer.close()

    print(f"\n✅ Extraction terminée ! {total_lignes:,} lignes extraites sans décalage.")

except Exception as e:
    print(f"❌ Erreur : {e}")

---

#### Vérification du dataset téléchargé

In [ ]:
# Vérification du format 

import pyarrow.parquet as pq

parquet_file = pq.ParquetFile("extraction_sas_sarl_2026.parquet")

print(f"📊 Nombre de lignes : {parquet_file.metadata.num_rows:,}")
print(f"📋 Nombre de colonnes : {parquet_file.metadata.num_columns}")

In [ ]:
import pandas as pd

# 1. Analyse de la structure via les métadonnées (0 RAM)
parquet_file = pq.ParquetFile("extraction_sas_sarl_2026.parquet")
all_cols = parquet_file.schema.names
total_cols = len(all_cols)

print(f"🔍 Audit du fichier : {total_cols} colonnes détectées.\n")

# 2. Fonction pour inspecter un segment spécifique
def inspecter_segment(debut, fin):
    segment = all_cols[debut:fin]
    # On lit les 5 premières lignes pour ce groupe de colonnes
    df_segment = parquet_file.read_row_group(0, columns=segment).to_pandas().head(5)
    
    # Nettoyage visuel des noms de colonnes (BOM)
    df_segment.columns = [c.replace('\ufeff', '') for c in df_segment.columns]
    
    print(f"--- Colonnes {debut} à {min(fin, total_cols)} ---")
    display(df_segment)

# --- À TOI DE JOUER : Change les index ici pour naviguer ---
# Pour voir les 20 premières :
inspecter_segment(0, 20)

# Pour voir la FIN (très important pour vérifier si c'est tronqué) :
inspecter_segment(total_cols-10, total_cols)

In [ ]:
# Nettoyage et récupération de la liste dans une variable
liste_finale = [c.replace('\ufeff', '') for c in all_cols]

# Affichage
print(liste_finale)

---